In [ ]:
import os
from google import genai

os.environ["GEMINI_API_KEY"] = "secret_key"

In [2]:
from google import genai

client = genai.Client()

for m in client.models.list():
    name = getattr(m, "name", "")
    if "gemini" in name.lower():
        print(name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1

In [3]:
from google import genai

client = genai.Client()

models = client.models.list()

for m in models:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [3]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with exactly: OK"
)

print(response.text)

OK


In [5]:
import os
from google import genai
from google.genai import types

BASE_DIR = r"C:\Docs\Lectures\Spring_2026\Adv_Computer_Architecture\project\tasks"
OUTPUT_DIR = r"C:\Docs\Lectures\Spring_2026\Adv_Computer_Architecture\project\test_3_gemini"
MODEL = "gemini-2.5-flash"
NUM_AGENTS = 2
TEMPERATURE = 0.5

# If GEMINI_API_KEY is already set in your notebook/session, this will use it.
# Otherwise you can pass api_key="..." here.
client = genai.Client()

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================
def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def save_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def clean_code(text):
    text = text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 3:
            text = parts[1]
            if "\n" in text:
                first_line, rest = text.split("\n", 1)
                if first_line.strip().lower() in {
                    "c", "cpp", "c++", "h", "hpp", "header"
                }:
                    text = rest
        text = text.strip()
    return text

def build_full_prompt(messages):
    parts = []
    for msg in messages:
        role = msg["role"].upper()
        parts.append(f"{role}:\n{msg['content']}")
    return "\n\n".join(parts)

def ask_agent(messages, user_content):
    messages.append({"role": "user", "content": user_content})

    full_prompt = build_full_prompt(messages)

    response = client.models.generate_content(
        model=MODEL,
        contents=full_prompt,
        config=types.GenerateContentConfig(
            temperature=TEMPERATURE
        )
    )

    reply = response.text.strip()
    messages.append({"role": "assistant", "content": reply})
    return reply

# =========================
# SINGLE AGENT SETUP
# =========================
system_prompt = """You are participating in a step-by-step study session.

You will receive materials one stage at a time.
You must only work on the current stage.
Do not jump ahead.

Rules:
- When asked to answer a quiz, return only the quiz answers.
- When asked to complete a header file, return only the final completed header file.
- Do not include markdown code fences.
- Do not include explanations.
- Do not include extra text like 'Here is the solution'.
"""

# =========================
# STATIC INPUTS
# =========================
mem1_path = os.path.join(BASE_DIR, "mem_model_1.md")
mem1_text = read_text(mem1_path)

mem2_path = os.path.join(BASE_DIR, "mem_model_2.md")
mem2_text = read_text(mem2_path)

# =========================
# RUN 200 AGENTS
# =========================
for agent_num in range(1, NUM_AGENTS + 1):
    agent_id = f"{MODEL}_{agent_num:03d}"
    agent_output_dir = os.path.join(OUTPUT_DIR, agent_id)
    os.makedirs(agent_output_dir, exist_ok=True)

    print(f"Starting {agent_id} ...")

    messages = [
        {"role": "system", "content": system_prompt}
    ]

    try:
        # =========================
        # STEP 1: MEMORY MODEL 1 QUIZ
        # =========================
        quiz1_prompt = f"""

Material:
{mem1_text}
"""
        quiz1_reply = ask_agent(messages, quiz1_prompt)
        save_text(os.path.join(agent_output_dir, "mem_model_1_quiz.txt"), quiz1_reply)

        # =========================
        # STEP 2: SPSC SC TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "spsc_sc_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "spsc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "spsc_sc_c", "spsc_queue.h"),
            clean_code(task_reply)
        )

        # =========================
        # STEP 3: MPMC SC TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "mpmc_sc_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "mpmc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "mpmc_sc_c", "mpmc_queue.h"),
            clean_code(task_reply)
        )

        # =========================
        # STEP 4: MEMORY MODEL 2 QUIZ
        # =========================
        quiz2_prompt = f"""

Material:
{mem2_text}
"""
        quiz2_reply = ask_agent(messages, quiz2_prompt)
        save_text(os.path.join(agent_output_dir, "mem_model_2_quiz.txt"), quiz2_reply)

        # =========================
        # STEP 5: SPSC RELAXED TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "spsc_relaxed_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "spsc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "spsc_relaxed_c", "spsc_queue.h"),
            clean_code(task_reply)
        )

        # =========================
        # STEP 6: MPMC RELAXED TASK
        # =========================
        task_dir = os.path.join(BASE_DIR, "mpmc_relaxed_c")
        readme_text = read_text(os.path.join(task_dir, "readme.md"))
        skeleton_text = read_text(os.path.join(task_dir, "mpmc_queue.h"))

        task_prompt = f"""

Task README:
{readme_text}

Header skeleton:
{skeleton_text}

"""
        task_reply = ask_agent(messages, task_prompt)
        save_text(
            os.path.join(agent_output_dir, "mpmc_relaxed_c", "mpmc_queue.h"),
            clean_code(task_reply)
        )

        print(f"Completed {agent_id}")

    except Exception as e:
        error_path = os.path.join(agent_output_dir, "error.txt")
        save_text(error_path, str(e))
        print(f"Failed {agent_id}: {e}")

print("All agent runs completed.")

Starting gemini-2.5-flash_001 ...
Completed gemini-2.5-flash_001
Starting gemini-2.5-flash_002 ...
Completed gemini-2.5-flash_002
All agent runs completed.
